# __Principio de inversión de dependencia__
En este principio los modulos de alto nivel no deben depender de las implementaciones de bajo nivel, ambos deben depender de abstracciones.

## Código sin emplear este principio.
Para esto utilizaremos un sistema de cobro de peaje(Alto Nivel) crea una instancia y depende directamente de un LectorTagRFIDFisico específico (Bajo Nivel). Si se cambia a lectura de placas por cámara, el código se rompe. El carril sigue instanciando directamente al lector.


In [ ]:
class LectorTagRFID:
    def leer_chip(self):
        return "TAG-XYZ-123"

    def emitir_pitido(self):
        print("Beep!")

class CarrilPeaje:
    def __init__(self):
        # Dependencia rígida de una clase concreta (Sin aplicar DIP)
        self.lector = LectorTagRFID()

    def procesar_vehiculo(self):
        self.lector.emitir_pitido()
        tag = self.lector.leer_chip()
        return f"Vehículo detectado con tag: {tag}"


carril_norte = CarrilPeaje()

#Simulamos el paso de un vehículo
resultado = carril_norte.procesar_vehiculo()

print(resultado)

Beep!
Vehículo detectado con tag: TAG-XYZ-123


## Código empleando este principio.

Ahora se implementará una interfaz de lectura inicial dada por el chip y la emisión de un sonido al hacerlo, luego se define dos clases para implementar la abstracción. Si se decide cambiar de proveedor de telepeaje o decides leer placas con cámaras en lugar de tags, la clase CarrilPeaje no sufre ninguna modificación.

In [4]:
from abc import ABC, abstractmethod


class ILector(ABC):
    @abstractmethod
    def leer_chip(self) -> str:
        pass

    @abstractmethod
    def emitir_pitido(self):
        pass


class LectorTagRFID(ILector):
    def leer_chip(self) -> str:
        return "TAG-XYZ-888"

    def emitir_pitido(self):
        print("Beep! (RFID)")

#Se puede crear otros lectores sin modificar el carril
class LectorCamaraPlaca(ILector):
    def leer_chip(self) -> str:
        return "PLACA-ABC-123"

    def emitir_pitido(self):
        print("¡Flash! (Cámara)")

#El carril ahora depende de la abstracción, no de un lector específico
class CarrilPeaje:
    def __init__(self, lector: ILector):
        # Inyección de dependencia
        self.lector = lector

    def procesar_vehiculo(self):
        self.lector.emitir_pitido()
        tag = self.lector.leer_chip()
        return f"Vehículo detectado con ID: {tag}"



lector_rfid = LectorTagRFID()
lector_camara = LectorCamaraPlaca()

# Inyectamos el lector RFID al primer carril
carril_norte = CarrilPeaje(lector_rfid)
print("--- Carril #1 ---")
print(carril_norte.procesar_vehiculo())

# Inyectamos el lector de cámara a otro carril (¡sin tocar el código de CarrilPeaje!)
carril_sur = CarrilPeaje(lector_camara)
print("\n--- Carril #2 ---")
print(carril_sur.procesar_vehiculo())

--- Carril #1 ---
Beep! (RFID)
Vehículo detectado con ID: TAG-XYZ-888

--- Carril #2 ---
¡Flash! (Cámara)
Vehículo detectado con ID: PLACA-ABC-123
